# 🔀 Flow Matching

This notebook builds deep intuition for:
1. **Continuous normalizing flows** — ODEs that transform distributions
2. **The flow matching idea** — learning vector fields without simulation
3. **Conditional flow matching** — the trick that makes training tractable
4. **Optimal transport paths** — the shortest way to move probability mass
5. **Linear interpolation** — the surprisingly powerful straight-line path
6. **From noise to data** — generating samples by following learned flows
7. **Why flow matching works** — connections to diffusion and beyond

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 1. From Discrete Flows to Continuous Flows

In Tutorial 17, we composed discrete transformation steps: $f_K \circ \cdots \circ f_1$.

What if we take infinitely many infinitesimal steps? We get an **ODE**:

$$\frac{dx}{dt} = v_\theta(x, t), \quad t \in [0, 1]$$

Think of it like a **river**: every point in space has a velocity at each time $t$.
Place a particle (noise sample) at $t=0$, and it flows along the velocity field to $t=1$ (data).

The collection of all particles transforms the noise distribution into the data distribution.

In [ ]:
# Visualize: a velocity field transporting particles

# Simple example: velocity field that moves a Gaussian to a target
# v(x, t) = (target - x) — always points toward the target

# Target: two clusters
targets = np.array([[-2, 1], [2, -1], [0, 2]])

# Start from noise
n_particles = 300
x0 = np.random.randn(n_particles, 2) * 0.8

# Assign each particle a random target
target_idx = np.random.randint(0, len(targets), n_particles)
particle_targets = targets[target_idx]

# Simulate: velocity = (target - current) / (1 - t)
n_steps = 50
trajectories = np.zeros((n_steps + 1, n_particles, 2))
trajectories[0] = x0

for k in range(n_steps):
    t = k / n_steps
    x_current = trajectories[k]
    # Linear interpolation velocity
    v = particle_targets - x0  # constant velocity = target - source
    trajectories[k+1] = x_current + v / n_steps

# Plot
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
times_to_show = [0, 15, 35, 50]
colors = ['steelblue', 'coral', 'green']

for i, t_idx in enumerate(times_to_show):
    ax = axes[i]
    for c_idx in range(len(targets)):
        mask = target_idx == c_idx
        ax.scatter(trajectories[t_idx, mask, 0], trajectories[t_idx, mask, 1],
                   s=8, alpha=0.5, c=colors[c_idx])
    
    # Show some trajectories
    for j in range(0, n_particles, 20):
        ax.plot(trajectories[:t_idx+1, j, 0], trajectories[:t_idx+1, j, 1],
                'k-', alpha=0.1, linewidth=0.5)
    
    ax.set_title(f't = {t_idx/n_steps:.2f}')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Particles Flowing from Noise to Data Along Straight Paths', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("Each particle follows its velocity field from noise (t=0) to data (t=1).")
print("Notice: the paths are STRAIGHT LINES — this is the optimal transport path!")

## 2. The Continuity Equation — Conservation of Probability

When particles move, probability density changes according to:

$$\frac{\partial p_t}{\partial t} + \nabla \cdot (p_t \cdot v_t) = 0$$

This is **conservation of probability** — probability is neither created nor destroyed, just moved.

It's the continuous-time version of the change-of-variables formula from normalizing flows.

In [ ]:
# 1D continuity equation: watching density evolve

# Start with N(0, 1), move it to N(3, 0.5) via linear interpolation
x = np.linspace(-4, 7, 500)

mu_0, sig_0 = 0, 1.0
mu_1, sig_1 = 3, 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: density at different times
colors = plt.cm.viridis(np.linspace(0, 1, 11))
for i, t in enumerate(np.linspace(0, 1, 11)):
    mu_t = (1-t) * mu_0 + t * mu_1
    sig_t = (1-t) * sig_0 + t * sig_1
    p_t = norm.pdf(x, mu_t, sig_t)
    axes[0].plot(x, p_t, color=colors[i], linewidth=1.5, alpha=0.8)

axes[0].set_xlabel('x')
axes[0].set_ylabel('p(x, t)')
axes[0].set_title('Density Evolving from N(0,1) to N(3, 0.25)')
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, 1))
plt.colorbar(sm, ax=axes[0], label='time t')
axes[0].grid(True, alpha=0.3)

# Right: particle trajectories
n_particles = 50
x0_particles = np.random.normal(mu_0, sig_0, n_particles)
x1_particles = np.random.normal(mu_1, sig_1, n_particles)

t_range = np.linspace(0, 1, 50)
for i in range(n_particles):
    xt = (1 - t_range) * x0_particles[i] + t_range * x1_particles[i]
    axes[1].plot(t_range, xt, 'b-', alpha=0.15, linewidth=0.8)

axes[1].set_xlabel('Time t')
axes[1].set_ylabel('x(t)')
axes[1].set_title('Individual Particle Trajectories')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🔑 The continuity equation ensures total probability is conserved.")
print("Density shifts and reshapes, but the integral always equals 1.")

## 3. The Problem — and the Solution

### The Problem with Continuous Normalizing Flows (CNFs)

Training a CNF requires **solving the ODE** during every training step:
- Forward ODE (many neural net evaluations)
- Backward ODE (for gradients)
- Each step is ~100x more expensive than a single forward pass

### The Flow Matching Solution

**Don't solve the ODE during training!** Instead:
1. Define a simple path from noise to data: $x_t = (1-t)x_0 + tx_1$
2. The velocity along this path is trivially $u_t = x_1 - x_0$
3. Train a neural network to predict this velocity: $\mathcal{L} = \|v_\theta(x_t, t) - (x_1 - x_0)\|^2$

**That's it!** Just regression. No ODE. No adversarial training. No ELBO.

In [ ]:
# The flow matching training recipe — step by step

# Target data: two moons
def make_moons(n, noise=0.08):
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.column_stack([np.cos(t), np.sin(t)])
    x2 = np.column_stack([1 - np.cos(t), -np.sin(t) + 0.5])
    return np.vstack([x1, x2]) + np.random.randn(n, 2) * noise

data = make_moons(2000)

# Visualize one training step
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

n_show = 20
x1_batch = data[:n_show]  # data samples
x0_batch = np.random.randn(n_show, 2)  # noise samples
t_batch = np.random.rand(n_show)  # random times

# Step 1: Sample pairs
axes[0].scatter(x0_batch[:, 0], x0_batch[:, 1], s=50, c='gray', marker='o', label='x₀ (noise)')
axes[0].scatter(x1_batch[:, 0], x1_batch[:, 1], s=50, c='coral', marker='*', label='x₁ (data)')
for i in range(n_show):
    axes[0].annotate('', xy=x1_batch[i], xytext=x0_batch[i],
                     arrowprops=dict(arrowstyle='->', color='blue', alpha=0.3))
axes[0].set_title('Step 1: Sample (x₀, x₁) pairs')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Step 2: Interpolate
xt_batch = (1 - t_batch[:, None]) * x0_batch + t_batch[:, None] * x1_batch
axes[1].scatter(xt_batch[:, 0], xt_batch[:, 1], s=50, c=t_batch, cmap='viridis', 
                edgecolors='black', linewidth=0.5)
axes[1].set_title('Step 2: Compute xₜ = (1-t)x₀ + tx₁')
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, 1))
plt.colorbar(sm, ax=axes[1], label='t')
axes[1].grid(True, alpha=0.3)

# Step 3: Target velocity
target_v = x1_batch - x0_batch  # This is the target!
axes[2].quiver(xt_batch[:, 0], xt_batch[:, 1], target_v[:, 0], target_v[:, 1],
               color='red', alpha=0.7, scale=20)
axes[2].scatter(xt_batch[:, 0], xt_batch[:, 1], s=30, c='black', zorder=5)
axes[2].set_title('Step 3: Target velocity u = x₁ - x₀')
axes[2].grid(True, alpha=0.3)

# Step 4: Loss
axes[3].text(0.5, 0.6, r'$\mathcal{L} = \|v_\theta(x_t, t) - (x_1 - x_0)\|^2$',
             fontsize=18, ha='center', va='center', transform=axes[3].transAxes)
axes[3].text(0.5, 0.35, 'Just MSE regression!\nNo ODE solving needed!',
             fontsize=13, ha='center', va='center', transform=axes[3].transAxes,
             color='green', fontweight='bold')
axes[3].set_title('Step 4: Train with simple loss')
axes[3].axis('off')

plt.suptitle('Flow Matching Training Recipe', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("🔑 The beauty of flow matching:")
print("1. Sample noise x₀ and data x₁")
print("2. Interpolate: xₜ = (1-t)x₀ + tx₁")
print("3. Target velocity is just: x₁ - x₀")
print("4. Train network to predict this velocity")
print("\nIt's THAT simple. No ODE. No adversarial game. Just regression.")

## 4. Training a Flow Matching Model

Let's implement the full training pipeline and see it work.

In [ ]:
# Full flow matching implementation

np.random.seed(42)

# Simple MLP for velocity prediction
class VelocityMLP:
    def __init__(self, hidden=64):
        scale = 0.3
        self.W1 = np.random.randn(3, hidden) * scale  # input: [x1, x2, t]
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, hidden) * scale
        self.b2 = np.zeros(hidden)
        self.W3 = np.random.randn(hidden, 2) * 0.1  # output: [v1, v2]
        self.b3 = np.zeros(2)
    
    def __call__(self, x, t):
        inp = np.column_stack([x, t.reshape(-1, 1) if t.ndim == 1 else t])
        h = np.tanh(inp @ self.W1 + self.b1)
        h = np.tanh(h @ self.W2 + self.b2)
        return h @ self.W3 + self.b3
    
    def params(self):
        return [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3]

model = VelocityMLP(hidden=48)

# Training
lr = 0.002
batch_size = 256
losses = []

for step in range(2000):
    # 1. Sample
    idx = np.random.randint(0, len(data), batch_size)
    x1 = data[idx]
    x0 = np.random.randn(batch_size, 2)
    t = np.random.rand(batch_size)
    
    # 2. Interpolate
    xt = (1 - t[:, None]) * x0 + t[:, None] * x1
    
    # 3. Target
    target = x1 - x0
    
    # 4. Predict & Loss
    pred = model(xt, t)
    loss = np.mean((pred - target)**2)
    losses.append(loss)
    
    # Stochastic gradient update (numerical)
    eps = 3e-4
    for param in model.params():
        flat = param.ravel()
        n_update = min(len(flat), 150)  # random subset for speed
        indices = np.random.choice(len(flat), n_update, replace=False)
        for j in indices:
            flat[j] += eps
            loss_p = np.mean((model(xt, t) - target)**2)
            flat[j] -= 2*eps
            loss_m = np.mean((model(xt, t) - target)**2)
            flat[j] += eps
            flat[j] -= lr * (loss_p - loss_m) / (2*eps)
    
    if step % 400 == 0:
        print(f"Step {step}: loss = {loss:.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

In [ ]:
# Generate samples by solving the ODE

def generate_samples(model, n_samples, n_steps=20):
    """Generate samples by solving dx/dt = v_theta(x, t) with Euler method."""
    x = np.random.randn(n_samples, 2)
    dt = 1.0 / n_steps
    trajectories = [x.copy()]
    
    for k in range(n_steps):
        t = np.full(n_samples, k * dt)
        v = model(x, t)
        x = x + v * dt
        trajectories.append(x.copy())
    
    return x, trajectories

samples, trajs = generate_samples(model, 1500, n_steps=30)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Generated vs Real
axes[0].scatter(data[:, 0], data[:, 1], s=3, alpha=0.3, c='steelblue', label='Real')
axes[0].scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.3, c='coral', label='Generated')
axes[0].set_title('Generated vs Real Data')
axes[0].legend()
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Trajectories
for i in range(40):
    path = np.array([t[i] for t in trajs])
    axes[1].plot(path[:, 0], path[:, 1], 'b-', alpha=0.15, linewidth=0.8)
axes[1].scatter(trajs[0][:40, 0], trajs[0][:40, 1], s=20, c='gray', zorder=5, label='Start (noise)')
axes[1].scatter(trajs[-1][:40, 0], trajs[-1][:40, 1], s=20, c='coral', zorder=5, label='End (data)')
axes[1].set_title('Sample Trajectories: Noise → Data')
axes[1].legend(fontsize=9)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

# Training loss
axes[2].plot(losses, alpha=0.7, color='steelblue')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('MSE Loss')
axes[2].set_title('Training Loss (smooth convergence!)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Flow Matching: Training Complete', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("🔑 Key observations:")
print("• Training loss converges smoothly (no adversarial oscillation!)")
print("• Trajectories are approximately straight (OT paths)")
print("• Generated samples match the data distribution well")

## 5. Optimal Transport — Why Straight Lines Are Best

The linear interpolation $x_t = (1-t)x_0 + tx_1$ corresponds to **optimal transport** —
the cheapest way to move probability mass from noise to data.

Why does this matter?
- **Straighter paths** → simpler velocity field → easier to learn
- **Straighter paths** → fewer ODE steps at inference → faster generation
- **Straighter paths** → less numerical error from discretization

Let's compare OT paths with curved (diffusion-style) paths.

In [ ]:
# Compare interpolation paths: OT (linear) vs VP (diffusion-style)

np.random.seed(42)

# Sample some (noise, data) pairs
n_pairs = 8
x0_pairs = np.random.randn(n_pairs, 2)
x1_pairs = data[np.random.choice(len(data), n_pairs)]

t_range = np.linspace(0, 0.999, 100)  # avoid t=1 for VP (singularity)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# OT (linear) paths
for i in range(n_pairs):
    path = np.array([(1-t)*x0_pairs[i] + t*x1_pairs[i] for t in t_range])
    axes[0].plot(path[:, 0], path[:, 1], '-', alpha=0.6, linewidth=1.5)
    axes[0].scatter(x0_pairs[i, 0], x0_pairs[i, 1], c='gray', s=30, zorder=5)
    axes[0].scatter(x1_pairs[i, 0], x1_pairs[i, 1], c='coral', s=30, zorder=5, marker='*')

axes[0].set_title('OT Path: xₜ = (1-t)x₀ + tx₁\n(Straight lines!)')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# VP (diffusion-style) paths
for i in range(n_pairs):
    path = np.array([np.sqrt(1-t**2)*x0_pairs[i] + t*x1_pairs[i] for t in t_range])
    axes[1].plot(path[:, 0], path[:, 1], '-', alpha=0.6, linewidth=1.5)
    axes[1].scatter(x0_pairs[i, 0], x0_pairs[i, 1], c='gray', s=30, zorder=5)
    axes[1].scatter(x1_pairs[i, 0], x1_pairs[i, 1], c='coral', s=30, zorder=5, marker='*')

axes[1].set_title('VP Path: xₜ = √(1-t²)x₀ + tx₁\n(Curved paths)')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

# Path straightness comparison
ot_straightness = []
vp_straightness = []

for _ in range(500):
    x0 = np.random.randn(2)
    x1 = data[np.random.randint(len(data))]
    
    displacement = np.sum((x1 - x0)**2)
    
    # OT path length
    ot_vel = x1 - x0  # constant
    ot_length = np.sum(ot_vel**2)  # integral of ||v||^2 dt = ||v||^2 * 1
    ot_straightness.append(displacement / ot_length)
    
    # VP path length
    dt = 0.001
    vp_length = 0
    for t in np.arange(0, 0.999, dt):
        vp_vel = -t / np.sqrt(1 - t**2 + 1e-8) * x0 + x1
        vp_length += np.sum(vp_vel**2) * dt
    vp_straightness.append(displacement / max(vp_length, 1e-8))

axes[2].boxplot([ot_straightness, vp_straightness], labels=['OT (linear)', 'VP (diffusion)'])
axes[2].set_ylabel('Straightness S (1 = perfectly straight)')
axes[2].set_title(f'Path Straightness\nOT: {np.mean(ot_straightness):.3f}, VP: {np.mean(vp_straightness):.3f}')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🔑 OT paths are perfectly straight (S=1) — maximum efficiency!")
print("VP paths are curved (S<1) — need more ODE steps to follow accurately.")
print(f"\nOT straightness: {np.mean(ot_straightness):.4f} (perfect)")
print(f"VP straightness: {np.mean(vp_straightness):.4f} (curved)")

## 6. How Many ODE Steps Do We Need?

One of the biggest practical advantages of flow matching with OT paths:
you need **far fewer steps** at inference compared to diffusion models.

Let's measure sample quality vs. number of Euler steps.

In [ ]:
# Effect of number of ODE steps on sample quality

step_counts = [1, 2, 5, 10, 20, 50]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for i, N in enumerate(step_counts):
    row, col = i // 3, i % 3
    ax = axes[row, col]
    
    samples_N, _ = generate_samples(model, 1000, n_steps=N)
    
    ax.scatter(data[:500, 0], data[:500, 1], s=2, alpha=0.2, c='steelblue', label='Real')
    ax.scatter(samples_N[:, 0], samples_N[:, 1], s=3, alpha=0.3, c='coral', label='Generated')
    ax.set_title(f'N = {N} Euler steps')
    ax.set_xlim(-2, 3)
    ax.set_ylim(-1.5, 2)
    if i == 0:
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Sample Quality vs Number of ODE Steps', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("🔑 With OT paths:")
print("• N=1 step: crude but already captures general shape")
print("• N=5 steps: reasonable quality")
print("• N=10-20 steps: excellent quality")
print("\nDiffusion models typically need 50-1000 steps for similar quality!")
print("This 5-50x speedup is a major practical advantage of flow matching.")

## 7. Visualizing the Learned Vector Field

Let's look at what the model actually learned — the velocity field $v_\theta(x, t)$ at different times.

In [ ]:
# Visualize learned vector field at different times

fig, axes = plt.subplots(2, 5, figsize=(22, 8))

# Grid for vector field
xx, yy = np.meshgrid(np.linspace(-3, 4, 18), np.linspace(-2.5, 3, 18))
grid = np.column_stack([xx.ravel(), yy.ravel()])

times = [0.0, 0.2, 0.4, 0.6, 0.8]

for i, t_val in enumerate(times):
    # Top row: vector field
    t_grid = np.full(len(grid), t_val)
    v = model(grid, t_grid)
    speed = np.sqrt(v[:, 0]**2 + v[:, 1]**2)
    
    axes[0, i].quiver(grid[:, 0], grid[:, 1], v[:, 0], v[:, 1],
                      speed, cmap='coolwarm', alpha=0.8, scale=25)
    axes[0, i].scatter(data[:200, 0], data[:200, 1], s=1, alpha=0.15, c='steelblue')
    axes[0, i].set_title(f'v(x, t={t_val})', fontsize=11)
    axes[0, i].set_xlim(-3, 4)
    axes[0, i].set_ylim(-2.5, 3)
    axes[0, i].set_aspect('equal')
    axes[0, i].grid(True, alpha=0.2)

# Bottom row: particle distribution at each time
n_vis = 2000
x_vis = np.random.randn(n_vis, 2)

for i, t_val in enumerate(times):
    # Evolve to time t
    x_at_t = np.random.randn(n_vis, 2)  # restart each time
    n_sub = max(1, int(t_val * 30))
    if n_sub > 0:
        dt = t_val / n_sub
        for k in range(n_sub):
            t_k = np.full(n_vis, k * dt)
            v = model(x_at_t, t_k)
            x_at_t = x_at_t + v * dt
    
    axes[1, i].scatter(x_at_t[:, 0], x_at_t[:, 1], s=1, alpha=0.3, c='coral')
    axes[1, i].scatter(data[:200, 0], data[:200, 1], s=1, alpha=0.1, c='steelblue')
    axes[1, i].set_title(f'Particles at t={t_val}')
    axes[1, i].set_xlim(-3, 4)
    axes[1, i].set_ylim(-2.5, 3)
    axes[1, i].set_aspect('equal')
    axes[1, i].grid(True, alpha=0.2)

plt.suptitle('Top: Learned Vector Field | Bottom: Particle Distribution Over Time', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("🔑 The vector field guides particles from Gaussian noise to the target distribution.")
print("At t=0, arrows point outward from the center toward data locations.")
print("As t→1, arrows become smaller as particles approach their destinations.")

## 8. The Complete Picture

### Flow Matching vs Other Generative Models

| Feature | Flow Matching | Diffusion | GANs | VAEs | Disc. Flows |
|---------|--------------|-----------|------|------|-------------|
| **Training** | Regression | Regression | Adversarial | ELBO | MLE |
| **Sampling** | ODE (5-20 steps) | SDE/ODE (50-1000) | 1 step | 1 step | 1 pass |
| **Likelihood** | Via ODE | Via SDE | None | ELBO | Exact |
| **Architecture** | Free | Free | Free | Enc+Dec | Invertible |
| **Quality** | Excellent | Excellent | Excellent | Good | Good |

### Why Flow Matching Won

1. **Simplest training**: just regression, no tricks needed
2. **Fastest sampling**: OT paths need far fewer steps than diffusion
3. **Most flexible**: no architectural constraints
4. **Best theory**: clean connection to optimal transport
5. **State-of-the-art results**: Stable Diffusion 3, Flux, and more use flow matching

### Key Equations Summary

| Concept | Equation |
|---------|----------|
| ODE | $dx/dt = v_\theta(x, t)$ |
| Linear path | $x_t = (1-t)x_0 + tx_1$ |
| Target velocity | $u_t = x_1 - x_0$ |
| CFM loss | $\|v_\theta(x_t, t) - (x_1 - x_0)\|^2$ |
| Euler step | $x_{t+\Delta t} = x_t + v_\theta \cdot \Delta t$ |